# Unit Tests
Run all backend tests from this notebook after each important backend change.

In [ ]:
import json
import os
import tempfile
import unittest
from unittest import mock

import backend.server as server
from backend.models import create_connection_state, create_match, create_user_session
from backend.protocol import ProtocolError, decode_message, encode_message
from frontend.net_utils import decode_message as front_decode_message, encode_message as front_encode_message
import sys
import types


In [ ]:
class DummySocket:
    def __init__(self):
        self.sent = []

    def sendall(self, data):
        self.sent.append(data)

    def shutdown(self, how):
        return None

    def close(self):
        return None


def parse_sent_messages(sock):
    parsed = []
    for raw in sock.sent:
        line = raw.decode('utf-8').strip()
        parsed.append(json.loads(line))
    return parsed


def message_types(sock):
    return [message['type'] for message in parse_sent_messages(sock)]


class TestProtocol(unittest.TestCase):
    def test_encode_message_has_newline_and_shape(self):
        raw = encode_message('LOGIN', {'username': 'Ali'})
        self.assertTrue(raw.endswith(b'\n'))
        parsed = json.loads(raw.decode('utf-8'))
        self.assertEqual(parsed['type'], 'LOGIN')
        self.assertEqual(parsed['payload'], {'username': 'Ali'})

    def test_decode_message_valid(self):
        parsed = decode_message(b'{"type":"WATCH_MATCH","payload":{}}')
        self.assertEqual(parsed, {'type': 'WATCH_MATCH', 'payload': {}})

    def test_decode_message_rejects_invalid_messages(self):
        with self.assertRaises(ProtocolError):
            decode_message(b'{"type":"LOGIN"')
        with self.assertRaises(ProtocolError):
            decode_message(b'{"payload":{}}')
        with self.assertRaises(ProtocolError):
            decode_message(b'{"type":"LOGIN","payload":[]}')




class TestFrontendNetUtils(unittest.TestCase):
    def test_frontend_encode_message_shape(self):
        raw = front_encode_message('PING', {'x': 1})
        self.assertTrue(raw.endswith(b'\n'))
        data = json.loads(raw.decode('utf-8'))
        self.assertEqual(data['type'], 'PING')
        self.assertEqual(data['payload'], {'x': 1})

    def test_frontend_decode_message_shape(self):
        parsed = front_decode_message(b'{"type":"PONG","payload":{"ok":true}}')
        self.assertEqual(parsed['type'], 'PONG')
        self.assertEqual(parsed['payload'], {'ok': True})

    def test_frontend_decode_message_rejects_bad_payload(self):
        with self.assertRaises(ValueError):
            front_decode_message(b'{"type":"BAD","payload":[]}')
class TestServer(unittest.TestCase):
    def setUp(self):
        server.STATE = create_connection_state()
        server.RUNNING.clear()
        self.original_db_path = server.HIGHSCORE_DB_PATH
        self.temp_db_path = os.path.join(tempfile.gettempdir(), 'python_arena_test_pie_highscores.sqlite')
        if os.path.exists(self.temp_db_path):
            os.remove(self.temp_db_path)
        server.HIGHSCORE_DB_PATH = self.temp_db_path
        server.HIGHSCORE_MEMORY_STORE.clear()
        server.initialize_highscore_store()

    def tearDown(self):
        server.HIGHSCORE_DB_PATH = self.original_db_path
        if os.path.exists(self.temp_db_path):
            os.remove(self.temp_db_path)

    def register_user(self, username, port):
        sock = DummySocket()
        session = create_user_session(sock, ('127.0.0.1', port))
        server.handle_login(session, {'username': username})
        return session, sock

    def test_login_uniqueness(self):
        _, first_sock = self.register_user('Ali', 5001)
        second_sock = DummySocket()
        second_session = create_user_session(second_sock, ('127.0.0.1', 5002))
        server.handle_login(second_session, {'username': 'Ali'})

        self.assertIn('LOGIN_OK', message_types(first_sock))
        self.assertIsNone(second_session['username'])
        self.assertEqual(parse_sent_messages(second_sock)[0]['type'], 'LOGIN_REJECT')

    def test_watch_transition_sets_spectator(self):
        session, sock = self.register_user('Ali', 5001)

        server.dispatch_message(session, {'type': 'WATCH_MATCH', 'payload': {}})
        self.assertIn('Ali', server.STATE['spectators'])

        types = message_types(sock)
        self.assertIn('WATCH_MATCH', types)

    def test_challenge_and_accept_create_match(self):
        ali_session, ali_sock = self.register_user('Ali', 5001)
        maya_session, maya_sock = self.register_user('Maya', 5002)

        server.handle_challenge_player(ali_session, {'target': 'Maya'})
        self.assertEqual(server.STATE['pending_challenges'].get('Maya'), 'Ali')
        self.assertIn('CHALLENGE_PLAYER', message_types(ali_sock))
        self.assertIn('CHALLENGE_RECEIVED', message_types(maya_sock))

        server.handle_challenge_accept(maya_session, {'from': 'Ali'})
        self.assertIsNotNone(server.STATE['active_match'])
        players = set(server.STATE['active_match']['players'])
        self.assertEqual(players, {'Ali', 'Maya'})
        self.assertIn('MATCH_START', message_types(ali_sock))
        self.assertIn('MATCH_START', message_types(maya_sock))

    def test_spectator_promoted_to_player_gets_single_player_start(self):
        ali_session, ali_sock = self.register_user('Ali', 5001)
        maya_session, maya_sock = self.register_user('Maya', 5002)

        server.dispatch_message(ali_session, {'type': 'WATCH_MATCH', 'payload': {}})
        server.dispatch_message(maya_session, {'type': 'WATCH_MATCH', 'payload': {}})
        self.assertIn('Ali', server.STATE['spectators'])
        self.assertIn('Maya', server.STATE['spectators'])

        server.handle_challenge_player(ali_session, {'target': 'Maya'})
        server.handle_challenge_accept(maya_session, {'from': 'Ali'})

        self.assertNotIn('Ali', server.STATE['spectators'])
        self.assertNotIn('Maya', server.STATE['spectators'])

        ali_starts = [m for m in parse_sent_messages(ali_sock) if m['type'] == 'MATCH_START']
        maya_starts = [m for m in parse_sent_messages(maya_sock) if m['type'] == 'MATCH_START']
        self.assertEqual(len(ali_starts), 1)
        self.assertEqual(len(maya_starts), 1)
        self.assertFalse(ali_starts[0]['payload']['spectator'])
        self.assertFalse(maya_starts[0]['payload']['spectator'])

    def test_input_updates_pending_direction(self):
        ali_session, _ = self.register_user('Ali', 5001)
        self.register_user('Maya', 5002)
        match, _ = server.create_and_start_match('Ali', 'Maya')
        self.assertIsNotNone(match)

        server.handle_input(ali_session, {'direction': 'UP'})
        self.assertEqual(server.STATE['active_match']['snakes']['Ali']['pending_direction'], 'UP')

        bad_sock = DummySocket()
        bad_session = create_user_session(bad_sock, ('127.0.0.1', 5010))
        bad_session['username'] = 'Ghost'
        server.handle_input(bad_session, {'direction': 'LEFT'})
        self.assertEqual(parse_sent_messages(bad_sock)[0]['type'], 'ERROR')

    def test_advance_tick_can_finish_on_timer(self):
        config = server.get_match_config()
        match = create_match(99, 'Ali', 'Maya', config)
        match['remaining_ticks'] = 1
        match['obstacles'] = []
        match['pies'] = []
        match['snakes']['Ali']['body'] = [(5, 5), (4, 5), (3, 5)]
        match['snakes']['Maya']['body'] = [(20, 5), (21, 5), (22, 5)]
        match['snakes']['Ali']['direction'] = 'RIGHT'
        match['snakes']['Ali']['pending_direction'] = 'RIGHT'
        match['snakes']['Maya']['direction'] = 'LEFT'
        match['snakes']['Maya']['pending_direction'] = 'LEFT'
        match['snakes']['Ali']['pies_collected'] = 4
        match['snakes']['Maya']['pies_collected'] = 2

        server.advance_match_one_tick(match, config)

        self.assertEqual(match['status'], 'ended')
        self.assertEqual(match['reason'], 'timer_end')
        self.assertEqual(match['winner'], 'Ali')

    def test_random_obstacle_shapes_max_five_and_away_from_corners(self):
        config = server.get_match_config()
        match = create_match(131, 'Ali', 'Maya', config)
        obstacles = set(match['obstacles'])

        self.assertTrue(obstacles)

        corner_margin = 4
        width = config['grid_width']
        height = config['grid_height']

        for x, y in obstacles:
            in_top_left = x < corner_margin and y < corner_margin
            in_top_right = x >= width - corner_margin and y < corner_margin
            in_bottom_left = x < corner_margin and y >= height - corner_margin
            in_bottom_right = x >= width - corner_margin and y >= height - corner_margin
            self.assertFalse(in_top_left or in_top_right or in_bottom_left or in_bottom_right)

        remaining = set(obstacles)
        components = []
        while remaining:
            start = remaining.pop()
            stack = [start]
            component = {start}
            while stack:
                cx, cy = stack.pop()
                for nx, ny in ((cx + 1, cy), (cx - 1, cy), (cx, cy + 1), (cx, cy - 1)):
                    if (nx, ny) in remaining:
                        remaining.remove((nx, ny))
                        component.add((nx, ny))
                        stack.append((nx, ny))
            components.append(component)

        self.assertLessEqual(len(components), 5)
        for component in components:
            self.assertIn(len(component), {2, 3, 4})

    def test_random_obstacles_can_spawn_during_match(self):
        config = server.get_match_config()
        config['obstacle_spawn_interval_ticks'] = 1
        config['obstacle_spawn_chance'] = 1.0
        config['max_obstacle_shapes'] = 2
        match = create_match(132, 'Ali', 'Maya', config)
        match['obstacle_shapes'] = []
        match['obstacles'] = []
        match['pies'] = [{'x': 15, 'y': 10, 'kind': 'green', 'value': config['pie_heal']}]

        with mock.patch('backend.server.random.random', return_value=0.0):
            spawned = server.maybe_spawn_match_obstacle(match, config)

        self.assertTrue(spawned)
        self.assertTrue(match['obstacles'])
        self.assertEqual(len(match['obstacle_shapes']), 1)
        self.assertEqual(len(match['obstacles']), len(match['obstacle_shapes'][0]))
        self.assertNotIn((15, 10), set(match['obstacles']))

    def test_disconnect_marks_match_ended(self):
        ali_session, _ = self.register_user('Ali', 5001)
        self.register_user('Maya', 5002)
        match, _ = server.create_and_start_match('Ali', 'Maya')
        self.assertEqual(match['status'], 'running')

        server.disconnect_session(ali_session)

        active = server.STATE['active_match']
        self.assertIsNotNone(active)
        self.assertEqual(active['status'], 'ended')
        self.assertEqual(active['reason'], 'player_disconnected')
        self.assertEqual(active['winner'], 'Maya')


    def test_collision_pause_flicker_and_resume_turn(self):
        config = server.get_match_config()
        match = create_match(120, 'Ali', 'Maya', config)
        match['remaining_ticks'] = 500
        match['pies'] = []
        match['obstacles'] = []

        ali = match['snakes']['Ali']
        maya = match['snakes']['Maya']

        ali['body'] = [(0, 5), (1, 5), (2, 5)]
        ali['direction'] = 'LEFT'
        ali['pending_direction'] = 'LEFT'

        maya['body'] = [(20, 10), (21, 10), (22, 10)]
        maya['direction'] = 'LEFT'
        maya['pending_direction'] = 'LEFT'

        server.advance_match_one_tick(match, config)

        self.assertEqual(ali['body'], [(0, 5), (1, 5), (2, 5)])
        self.assertEqual(ali['stun_ticks_remaining'], config['collision_pause_ticks'])
        self.assertEqual(ali['resume_direction'], 'DOWN')
        self.assertEqual(ali['health'], config['starting_health'] - config['wall_damage'])

        for _ in range(config['collision_pause_ticks'] - 1):
            before = list(ali['body'])
            server.advance_match_one_tick(match, config)
            self.assertEqual(ali['body'], before)

        self.assertEqual(ali['stun_ticks_remaining'], 1)

        server.advance_match_one_tick(match, config)
        self.assertEqual(ali['stun_ticks_remaining'], 0)
        self.assertEqual(ali['direction'], 'DOWN')

        before_move = list(ali['body'])
        server.advance_match_one_tick(match, config)
        self.assertNotEqual(ali['body'][0], before_move[0])


    def test_collision_resume_uses_last_valid_input_after_flicker(self):
        config = server.get_match_config()
        match = create_match(121, 'Ali', 'Maya', config)
        match['remaining_ticks'] = 500
        match['pies'] = []
        match['obstacles'] = []

        ali = match['snakes']['Ali']
        maya = match['snakes']['Maya']

        ali['body'] = [(0, 5), (1, 5), (2, 5)]
        ali['direction'] = 'LEFT'
        ali['pending_direction'] = 'LEFT'

        maya['body'] = [(20, 10), (21, 10), (22, 10)]
        maya['direction'] = 'LEFT'
        maya['pending_direction'] = 'LEFT'

        server.advance_match_one_tick(match, config)
        self.assertGreater(ali['stun_ticks_remaining'], 0)

        # queue a valid turn while Ali is still flickering.
        ali['pending_direction'] = 'UP'

        for _ in range(config['collision_pause_ticks']):
            server.advance_match_one_tick(match, config)
            if ali['stun_ticks_remaining'] == 0:
                break

        self.assertEqual(ali['stun_ticks_remaining'], 0)
        self.assertEqual(ali['direction'], 'UP')


    def test_orange_pie_adds_five_seconds(self):
        config = server.get_match_config()
        match = create_match(122, 'Ali', 'Maya', config)
        match['obstacles'] = []
        match['pies'] = [{'x': 3, 'y': 10, 'kind': 'orange', 'value': 1}]

        ali = match['snakes']['Ali']
        ali['body'] = [(2, 10), (1, 10), (0, 10)]
        ali['direction'] = 'RIGHT'
        ali['pending_direction'] = 'RIGHT'

        before_ticks = match['remaining_ticks']
        before_pies = ali['pies_collected']
        server.advance_match_one_tick(match, config)
        self.assertEqual(match['remaining_ticks'], before_ticks - 1 + 5 * config['tick_rate'])
        self.assertEqual(ali['pies_collected'], before_pies + 1)


    def test_green_pie_restores_health(self):
        config = server.get_match_config()
        match = create_match(123, 'Ali', 'Maya', config)
        match['obstacles'] = []
        match['pies'] = [{'x': 3, 'y': 10, 'kind': 'green', 'value': config['pie_heal']}]

        ali = match['snakes']['Ali']
        ali['body'] = [(2, 10), (1, 10), (0, 10)]
        ali['direction'] = 'RIGHT'
        ali['pending_direction'] = 'RIGHT'
        ali['health'] = 50

        server.advance_match_one_tick(match, config)
        self.assertEqual(ali['health'], min(config['starting_health'], 50 + config['pie_heal']))

    def test_blue_pie_slows_opponent_for_short_duration(self):
        config = server.get_match_config()
        match = create_match(124, 'Ali', 'Maya', config)
        match['obstacles'] = []
        match['pies'] = [{'x': 3, 'y': 10, 'kind': 'blue', 'value': 0}]

        ali = match['snakes']['Ali']
        maya = match['snakes']['Maya']
        ali['body'] = [(2, 10), (1, 10), (0, 10)]
        ali['direction'] = 'RIGHT'
        ali['pending_direction'] = 'RIGHT'

        maya['body'] = [(20, 10), (21, 10), (22, 10)]
        maya['direction'] = 'LEFT'
        maya['pending_direction'] = 'LEFT'

        server.advance_match_one_tick(match, config)
        self.assertGreater(maya['slow_ticks_remaining'], 0)
        self.assertEqual(maya['move_interval_ticks'], config['slowed_move_interval_ticks'])

        head_before = maya['body'][0]
        server.advance_match_one_tick(match, config)
        self.assertEqual(maya['body'][0], head_before)

        server.advance_match_one_tick(match, config)
        self.assertNotEqual(maya['body'][0], head_before)


    def test_purple_pie_grows_snake_one_block(self):
        config = server.get_match_config()
        match = create_match(125, 'Ali', 'Maya', config)
        match['obstacles'] = []
        match['pies'] = [{'x': 3, 'y': 10, 'kind': 'purple', 'value': 0}]

        ali = match['snakes']['Ali']
        ali['body'] = [(2, 10), (1, 10), (0, 10)]
        ali['direction'] = 'RIGHT'
        ali['pending_direction'] = 'RIGHT'

        before_len = len(ali['body'])
        server.advance_match_one_tick(match, config)
        server.advance_match_one_tick(match, config)
        self.assertEqual(len(ali['body']), before_len + 1)

    def test_highscore_sqlite_error_falls_back_to_memory(self):
        original_connect = server.sqlite3.connect

        def failing_connect(*args, **kwargs):
            raise server.sqlite3.OperationalError('database is locked')

        server.sqlite3.connect = failing_connect
        try:
            high_score, is_new = server.update_and_get_high_score('Ali', 4)
            self.assertEqual(high_score, 4)
            self.assertTrue(is_new)
            self.assertEqual(server.HIGHSCORE_STORE_MODE, 'memory')
        finally:
            server.sqlite3.connect = original_connect

    def test_game_over_payload_has_pie_stats_and_high_score_labels(self):
        config = server.get_match_config()
        match = create_match(130, 'Ali', 'Maya', config)

        match['snakes']['Ali']['pies_collected'] = 3
        match['snakes']['Maya']['pies_collected'] = 1
        payload_one = server.build_game_over_payload(match)

        self.assertEqual(payload_one['pie_stats']['Ali']['pies_collected'], 3)
        self.assertEqual(payload_one['pie_stats']['Ali']['high_score'], 3)
        self.assertEqual(payload_one['pie_stats']['Ali']['high_score_label'], 'new high score')
        self.assertEqual(payload_one['pie_stats']['Maya']['high_score_label'], 'new high score')

        match['snakes']['Ali']['pies_collected'] = 2
        match['snakes']['Maya']['pies_collected'] = 4
        payload_two = server.build_game_over_payload(match)

        self.assertEqual(payload_two['pie_stats']['Ali']['high_score'], 3)
        self.assertEqual(payload_two['pie_stats']['Ali']['high_score_label'], 'high score: 3')
        self.assertEqual(payload_two['pie_stats']['Maya']['high_score'], 4)
        self.assertEqual(payload_two['pie_stats']['Maya']['high_score_label'], 'new high score')

    def test_online_users_includes_active_match_metadata(self):
        _, ali_sock = self.register_user('Ali', 5001)
        self.register_user('Maya', 5002)

        match, error = server.create_and_start_match('Ali', 'Maya')
        self.assertIsNone(error)
        self.assertIsNotNone(match)

        ali_online_messages = [m for m in parse_sent_messages(ali_sock) if m['type'] == 'ONLINE_USERS']
        self.assertTrue(ali_online_messages)

        ali_last = ali_online_messages[-1]['payload']
        self.assertIn('active_matches', ali_last)
        self.assertEqual(len(ali_last['active_matches']), 1)
        active_match = ali_last['active_matches'][0]
        self.assertEqual(active_match['id'], match['id'])
        self.assertEqual(set(active_match['players']), {'Ali', 'Maya'})

    def test_watch_match_uses_selected_match_id(self):
        ali_session, ali_sock = self.register_user('Ali', 5001)
        self.register_user('Maya', 5002)

        match, _ = server.create_and_start_match('Ali', 'Maya')
        server.dispatch_message(ali_session, {'type': 'WATCH_MATCH', 'payload': {'match_id': match['id']}})

        watch_messages = [m for m in parse_sent_messages(ali_sock) if m['type'] == 'WATCH_MATCH']
        start_messages = [m for m in parse_sent_messages(ali_sock) if m['type'] == 'MATCH_START']
        self.assertTrue(watch_messages)
        self.assertTrue(start_messages)
        self.assertTrue(start_messages[-1]['payload']['spectator'])


    def test_cheer_message_is_added_to_match_chat(self):
        ali_session, _ = self.register_user('Ali', 5001)
        self.register_user('Maya', 5002)
        match, _ = server.create_and_start_match('Ali', 'Maya')

        server.dispatch_message(ali_session, {'type': 'CHEER', 'payload': {'text': 'gg'}})

        self.assertTrue(match['cheers'])
        self.assertEqual(match['cheers'][-1]['from'], 'Ali')
        self.assertEqual(match['cheers'][-1]['text'], 'gg')

    def test_build_match_payload_includes_cheers(self):
        config = server.get_match_config()
        match = create_match(150, 'Ali', 'Maya', config)
        match['cheers'] = [{'from': 'Ali', 'text': 'go green'}]

        payload = server.build_match_state_payload(match)
        self.assertIn('cheers', payload)
        self.assertEqual(payload['cheers'][0]['text'], 'go green')


    def test_waiting_message_adds_user_to_waiting_queue(self):
        session, sock = self.register_user('Ali', 5001)

        server.dispatch_message(session, {'type': 'WAITING', 'payload': {}})

        self.assertIn('Ali', server.STATE['waiting_players'])
        self.assertIn('WAITING', message_types(sock))

    def test_challenge_sets_challenger_waiting(self):
        ali_session, ali_sock = self.register_user('Ali', 5001)
        self.register_user('Maya', 5002)

        server.handle_challenge_player(ali_session, {'target': 'Maya'})

        self.assertIn('Ali', server.STATE['waiting_players'])
        types = message_types(ali_sock)
        self.assertIn('WAITING', types)
        self.assertIn('CHALLENGE_PLAYER', types)




class TestFrontendUI(unittest.TestCase):
    def load_frontend_ui(self):
        fake_pygame = types.SimpleNamespace()
        fake_pygame.SRCALPHA = 1
        sys.modules['pygame'] = fake_pygame

        import importlib
        module = importlib.import_module('frontend.ui')
        module = importlib.reload(module)
        return module

    def test_button_state_priority(self):
        fui = self.load_frontend_ui()
        self.assertEqual(fui.get_button_state(True, True), fui.STATE_PRESSED)
        self.assertEqual(fui.get_button_state(True, False), fui.STATE_HOVER)
        self.assertEqual(fui.get_button_state(False, True), fui.STATE_IDLE)

    def test_clamp_value_bounds(self):
        fui = self.load_frontend_ui()
        self.assertEqual(fui.clamp_value(-3, 0, 10), 0)
        self.assertEqual(fui.clamp_value(13, 0, 10), 10)
        self.assertEqual(fui.clamp_value(5, 0, 10), 5)

    def test_button_click_requires_press_start_inside(self):
        fui = self.load_frontend_ui()

        class FakeRect:
            def collidepoint(self, pos):
                return pos == (1, 1)

        button = fui.UIButton.__new__(fui.UIButton)
        button.rect = FakeRect()
        button.state = fui.STATE_IDLE
        button.was_down = False
        button.pressed_inside = False

        self.assertFalse(button.update((0, 0), True))
        self.assertFalse(button.update((1, 1), False))

        self.assertFalse(button.update((1, 1), True))
        self.assertTrue(button.update((1, 1), False))


    def test_multiply_brightness_uses_additive_blend_for_brightening(self):
        fui = self.load_frontend_ui()

        class FakeOverlay:
            def __init__(self, size):
                self.size = size
                self.fill_color = None

            def fill(self, color):
                self.fill_color = color

        class FakeSurface:
            def __init__(self, size):
                self.size = size
                self.blit_calls = []

            def copy(self):
                return FakeSurface(self.size)

            def get_size(self):
                return self.size

            def blit(self, overlay, pos, special_flags=None):
                self.blit_calls.append({
                    'overlay': overlay,
                    'pos': pos,
                    'special_flags': special_flags,
                })

        def fake_surface_factory(size, _flags):
            return FakeOverlay(size)

        fui.pygame.BLEND_RGBA_ADD = 200
        fui.pygame.BLEND_RGBA_MULT = 100
        fui.pygame.Surface = fake_surface_factory

        base = FakeSurface((4, 4))
        brightened = fui.multiply_brightness(base, 1.2)

        self.assertEqual(len(brightened.blit_calls), 1)
        call = brightened.blit_calls[0]
        self.assertEqual(call['special_flags'], fui.pygame.BLEND_RGBA_ADD)
        self.assertGreater(call['overlay'].fill_color[0], 0)

class TestFrontendClientConnectionLifecycle(unittest.TestCase):
    def load_frontend_client(self):
        fake_pygame = types.SimpleNamespace()
        fake_pygame.QUIT = 256
        fake_pygame.KEYDOWN = 768
        fake_pygame.K_TAB = 9
        fake_pygame.K_RETURN = 13
        fake_pygame.K_BACKSPACE = 8
        fake_pygame.K_UP = 273
        fake_pygame.K_DOWN = 274
        fake_pygame.K_LEFT = 276
        fake_pygame.K_RIGHT = 275
        fake_pygame.K_ESCAPE = 27
        fake_pygame.K_c = ord('c')
        fake_pygame.K_a = ord('a')
        fake_pygame.K_w = ord('w')
        fake_pygame.K_v = ord('v')
        fake_pygame.K_l = ord('l')
        sys.modules['pygame'] = fake_pygame

        import importlib
        module = importlib.import_module('frontend.client')
        module = importlib.reload(module)
        return module

    def test_stale_disconnect_event_is_ignored(self):
        fclient = self.load_frontend_client()
        state = fclient.create_client_state()
        state['connection_id'] = 2
        state['screen'] = fclient.SCREEN_LOBBY

        state['network_queue'].put({'connection_id': 1, 'message': {'type': 'DISCONNECTED', 'payload': {}}})
        fclient.process_network_queue(state)

        self.assertEqual(state['screen'], fclient.SCREEN_LOBBY)

    def test_close_connection_clears_receive_buffer(self):
        fclient = self.load_frontend_client()
        state = fclient.create_client_state()
        state['recv_buffer'] = b'partial-frame'

        fclient.close_connection(state)

        self.assertEqual(state['recv_buffer'], b'')


    def test_keyboard_shortcuts_trigger_matching_button_feedback(self):
        fclient = self.load_frontend_client()
        state = fclient.create_client_state()

        feedback_calls = []

        def record_feedback(_state, action_name):
            feedback_calls.append((_state['screen'], action_name))

        fclient.press_button_feedback = record_feedback

        state['screen'] = fclient.SCREEN_CONNECT
        fclient.handle_connect_screen_event(state, types.SimpleNamespace(type=fclient.pygame.KEYDOWN, key=fclient.pygame.K_RETURN, unicode=''))

        state['screen'] = fclient.SCREEN_USERNAME
        fclient.handle_username_screen_event(state, types.SimpleNamespace(type=fclient.pygame.KEYDOWN, key=fclient.pygame.K_RETURN, unicode=''))

        state['screen'] = fclient.SCREEN_LOBBY
        state['self_name'] = 'Ali'
        state['online_users'] = ['Ali', 'Maya']
        state['selected_user_index'] = 0
        state['pending_challenger'] = 'Maya'

        fclient.handle_lobby_screen_event(state, types.SimpleNamespace(type=fclient.pygame.KEYDOWN, key=fclient.pygame.K_c, unicode='c'))
        fclient.handle_lobby_screen_event(state, types.SimpleNamespace(type=fclient.pygame.KEYDOWN, key=fclient.pygame.K_a, unicode='a'))
        fclient.handle_lobby_screen_event(state, types.SimpleNamespace(type=fclient.pygame.KEYDOWN, key=fclient.pygame.K_v, unicode='v'))

        state['screen'] = fclient.SCREEN_GAME_OVER
        fclient.handle_game_over_screen_event(state, types.SimpleNamespace(type=fclient.pygame.KEYDOWN, key=fclient.pygame.K_l, unicode='l'))

        expected = [
            (fclient.SCREEN_CONNECT, 'connect'),
            (fclient.SCREEN_USERNAME, 'login'),
            (fclient.SCREEN_LOBBY, 'challenge'),
            (fclient.SCREEN_LOBBY, 'accept'),
            (fclient.SCREEN_LOBBY, 'watch'),
            (fclient.SCREEN_GAME_OVER, 'to_lobby'),
        ]
        self.assertEqual(feedback_calls, expected)


    def test_online_users_updates_active_matches_list(self):
        fclient = self.load_frontend_client()
        state = fclient.create_client_state()
        state['selected_match_index'] = 5

        fclient.handle_server_message(state, {
            'type': 'ONLINE_USERS',
            'payload': {
                'users': ['Ali', 'Maya'],
                'active_matches': [{'id': 7, 'players': ['Ali', 'Maya'], 'status': 'running'}],
            },
        })

        self.assertEqual(state['active_matches'][0]['id'], 7)
        self.assertEqual(state['selected_match_index'], 0)

    def test_lobby_watch_sends_selected_match_id(self):
        fclient = self.load_frontend_client()
        state = fclient.create_client_state()
        state['screen'] = fclient.SCREEN_LOBBY
        state['active_matches'] = [{'id': 11, 'players': ['Ali', 'Maya'], 'status': 'running'}]
        state['selected_match_index'] = 0

        sent = []

        def fake_send(_state, message_type, payload=None):
            sent.append((message_type, payload or {}))

        fclient.send_to_server = fake_send
        fclient.handle_lobby_screen_event(state, types.SimpleNamespace(type=fclient.pygame.KEYDOWN, key=fclient.pygame.K_v, unicode='v'))

        self.assertEqual(sent[-1], ('WATCH_MATCH', {'match_id': 11}))


    def test_player_number_key_sends_quick_chat(self):
        fclient = self.load_frontend_client()
        state = fclient.create_client_state()
        state['screen'] = fclient.SCREEN_GAME
        state['is_spectator'] = False

        sent = []

        def fake_send(_state, message_type, payload=None):
            sent.append((message_type, payload or {}))

        fclient.send_to_server = fake_send
        event = types.SimpleNamespace(type=fclient.pygame.KEYDOWN, key=0, unicode='2')
        fclient.handle_game_screen_event(state, event)

        self.assertEqual(sent[-1], ('CHEER', {'text': 'go blue'}))


    def test_spectator_chat_typing_and_enter_send(self):
        fclient = self.load_frontend_client()
        state = fclient.create_client_state()
        state['screen'] = fclient.SCREEN_GAME
        state['is_spectator'] = True
        state['chat_input'] = ''

        sent = []

        def fake_send(_state, message_type, payload=None):
            sent.append((message_type, payload or {}))

        fclient.send_to_server = fake_send

        fclient.handle_game_screen_event(state, types.SimpleNamespace(type=fclient.pygame.KEYDOWN, key=0, unicode='h'))
        fclient.handle_game_screen_event(state, types.SimpleNamespace(type=fclient.pygame.KEYDOWN, key=0, unicode='i'))
        fclient.handle_game_screen_event(state, types.SimpleNamespace(type=fclient.pygame.KEYDOWN, key=fclient.pygame.K_RETURN, unicode=''))

        self.assertEqual(sent[-1], ('CHEER', {'text': 'hi'}))
        self.assertEqual(state['chat_input'], '')


    def test_draw_menu_background_uses_fallback_when_image_missing(self):
        fclient = self.load_frontend_client()

        class FakeScreen:
            def __init__(self):
                self.filled = None
                self.blit_calls = 0

            def fill(self, color):
                self.filled = color

            def blit(self, *_args, **_kwargs):
                self.blit_calls += 1

        screen = FakeScreen()
        state = {'menu_background': None}

        fclient.draw_menu_background(screen, state)

        self.assertEqual(screen.filled, fclient.BOARD_BG)
        self.assertEqual(screen.blit_calls, 0)




    def test_waiting_message_updates_lobby_info(self):
        fclient = self.load_frontend_client()
        state = fclient.create_client_state()
        fclient.handle_server_message(state, {'type': 'WAITING', 'payload': {'status': 'queued'}})
        self.assertEqual(state['lobby_info'], 'Waiting for challenge response')


    def test_get_scaled_surface_returns_none_for_missing_source(self):
        fclient = self.load_frontend_client()
        state = {'scaled_surface_cache': {}}
        value = fclient.get_scaled_surface(state, 'any', None, 10, 10)
        self.assertIsNone(value)


In [ ]:
suite = unittest.TestSuite()
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TestProtocol))
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TestFrontendNetUtils))
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TestFrontendUI))
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TestFrontendClientConnectionLifecycle))
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TestServer))

runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

if not result.wasSuccessful():
    raise AssertionError('Some unit tests failed')
